# Session 1b: Embedding Explorer

**Course:** Language Models: ML Basics to Modern AI (BTU Cottbus, M.Sc. AI seminar)
**Session:** 1 of 4, notebook 1b of 3
**Lecture reference:** Lecture on word vectors, distributional semantics, and positional encoding.

## Learning objectives

By the end of this notebook you should be able to:

- Explain cosine similarity geometrically and apply it to compare word vectors.
- Predict and interpret the outcome of analogy arithmetic (e.g. `king - man + woman`) and recognise where it succeeds and where it fails.
- Identify gender and other social biases encoded in static word embeddings and articulate the source of those biases.
- Implement a `TokenEmbedder` module that sums a token embedding with a sinusoidal positional encoding, producing the tensor shape the transformer expects.
- Compare sinusoidal and learned positional encodings and justify the choice in a given setting.

The notebook accompanies the lecture on word vectors and positional encoding. It does not re-derive the theory. It makes the theory concrete by inspecting a trained embedding space and by building the input layer used in the next notebook.


## §1 Primer: vectors as meaning, and why positions are added separately

A transformer never sees a word. It sees a vector. Two questions follow. What does that vector encode, and where does the information about word order go?

**Distributional semantics.** A word is defined by the company it keeps (Firth, Harris, Mikolov, Pennington). Words appearing in similar contexts get similar vectors. The geometry tracks intuitive meaning: *doctor* and *physician* sit close, *doctor* and *bicycle* sit far apart. Linear directions encode relational structure: the vector from *man* to *woman* is approximately parallel to the one from *king* to *queen*, which gives the analogy `king - man + woman ≈ queen`.

**Cosine similarity** is the standard comparison in this setting. For vectors $a$ and $b$,

$$\cos(a, b) = \frac{a \cdot b}{\|a\| \, \|b\|}.$$

The output is the cosine of the angle between $a$ and $b$, in $[-1, 1]$: one for the same direction, zero for orthogonal, minus one for opposite. Magnitude is divided out. That matters because vector norms from co-occurrence statistics correlate with word frequency, which would otherwise dominate a raw dot product.

**Static versus contextual embeddings.** GloVe and word2vec assign each word one vector. *Bank* (river) and *bank* (savings) share the same point. Transformers fix this: the output of an attention block depends on every other token, so the same surface form produces different vectors in different sentences. Static embeddings remain useful pedagogically because the geometry is small enough to inspect by hand.

**Why positions are added separately.** A bag of word vectors loses ordering. *Dog bites man* and *man bites dog* produce the same multiset. Recurrent networks recover order from their sequential computation. Transformers process all positions in parallel, so position has to enter through the input. The standard recipe gives each position $t$ its own vector $p_t$ of the same dimension as the token embedding $x_t$ and forms the input as $x_t + p_t$. Adding (rather than concatenating) keeps the dimension fixed and lets the model allocate parts of the vector to content and parts to position; it works in practice and adds no parameters above.

**Sinusoidal versus learned positional encoding.** Any function $t \mapsto p_t$ could in principle play the role of the position vector. The original transformer picks a specific deterministic mix of sines and cosines because it buys two properties that a naive choice would not. First, every position is fully determined by a fixed formula, so a model trained on sequences of length 512 already knows what $p_{1024}$ looks like and can in principle extrapolate beyond training length. Second, the relative position $p_{t+k}$ is a linear function of $p_t$, which gives attention a clean basis for reasoning about offsets like "two positions back" or "five positions ahead" without ever seeing the absolute index. The formula is

$$p_{t, 2i} = \sin\!\left(\frac{t}{10000^{2i / d}}\right), \quad p_{t, 2i+1} = \cos\!\left(\frac{t}{10000^{2i / d}}\right),$$

a stack of sine and cosine waves whose frequencies span several orders of magnitude (from one cycle per token at the highest-frequency dimension to one cycle per ten-thousand tokens at the lowest). Different frequency bands let the model encode both fine and coarse position information in one vector.

Learned positional embeddings (an entry in `nn.Embedding(max_len, d_model)`) are the alternative. The position vector becomes a parameter, trained jointly with everything else. Learned embeddings fit the training distribution more tightly but cannot extrapolate past `max_len`. Pick sinusoidal when inference sequences may exceed training length, or when zero extra parameters matters. Pick learned when the maximum length is fixed and you have data. Modern systems often use neither, preferring relative schemes (T5 bias terms, RoPE) that inject position inside the attention computation rather than at the input.

In §4 you implement the cosine primitive and the input layer that sums token and positional vectors. That layer is the entry point of the `TinyGPT` model in 1c.


In [ ]:
"""§2 Setup: imports, random seed, device selection."""

import math
import warnings

import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

warnings.filterwarnings("ignore")

SEED = 0
np.random.seed(SEED)
torch.manual_seed(SEED)

# This notebook is CPU-friendly. GPU is not required for any of the cells below.
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch {torch.__version__} on {DEVICE}")


## §3 Guided exploration: the embedding explorer

The widget below has three parts. Part 1 takes a word and shows its ten nearest neighbours under cosine similarity in the GloVe space. Part 2 runs the classic analogy arithmetic `pos1 + pos2 - neg` and reports the closest words to the resulting vector. Part 3 sweeps a fixed list of professions along the gender axis and surfaces the asymmetries the embedding has absorbed from its training corpus.

The vectors come from GloVe (`glove-wiki-gigaword-100`, 400k words, 100 dimensions, trained on Wikipedia plus Gigaword). The download is roughly 128 MB. If the download fails (no network, a sandboxed environment), the cell falls back to a tiny synthetic dictionary of about twenty hand-picked words with deterministic random vectors. The widgets still work in that mode, but the analogies are meaningless; the cell prints a clearly marked notice when this happens.


In [ ]:
"""§3 Part 1: nearest neighbours in embedding space.

Loads GloVe via gensim if possible. Falls back to a deterministic synthetic
dictionary so the cell never raises in offline or sandboxed environments.
The fallback object exposes the same surface (`key_to_index`,
`most_similar`) that the three widget parts use, so the downstream cells
do not need to special-case the mode.
"""

import html as html_module

import ipywidgets as widgets
from IPython.display import HTML, display


class _SyntheticEmbedding:
    """Thin wrapper exposing the gensim KeyedVectors surface used by the widgets.

    The vectors are deterministic (fixed seed) but random, so neighbour rankings
    are arbitrary. The point is to keep the UI runnable offline.
    """

    # A small hand-picked vocabulary covering the widget's defaults and probes.
    _WORDS = [
        "king", "queen", "man", "woman", "boy", "girl",
        "paris", "france", "berlin", "germany", "tokyo", "japan",
        "doctor", "nurse", "engineer", "teacher", "professor",
        "scientist", "programmer", "artist", "lawyer", "therapist",
        "surgeon", "receptionist", "pilot", "secretary", "athlete",
        "rich", "poor", "young", "old",
    ]

    def __init__(self, dim: int = 32):
        rng = np.random.default_rng(0)
        self._vocab = list(dict.fromkeys(self._WORDS))  # dedup, preserve order
        self._vectors = rng.standard_normal((len(self._vocab), dim)).astype(np.float32)
        # L2-normalise so cosine similarity reduces to a dot product.
        norms = np.linalg.norm(self._vectors, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        self._vectors = self._vectors / norms
        self.key_to_index = {w: i for i, w in enumerate(self._vocab)}

    def __len__(self) -> int:
        return len(self._vocab)

    def __contains__(self, word: str) -> bool:
        return word in self.key_to_index

    def __getitem__(self, word: str) -> np.ndarray:
        return self._vectors[self.key_to_index[word]]

    def most_similar(self, positive=None, negative=None, topn: int = 10):
        if isinstance(positive, str):
            positive = [positive]
        positive = positive or []
        negative = negative or []
        unknown = [w for w in list(positive) + list(negative) if w not in self.key_to_index]
        if unknown:
            raise KeyError(unknown[0])
        target = np.zeros(self._vectors.shape[1], dtype=np.float32)
        for w in positive:
            target = target + self[w]
        for w in negative:
            target = target - self[w]
        norm = np.linalg.norm(target)
        if norm == 0:
            return []
        target = target / norm
        sims = self._vectors @ target
        excluded = set(positive) | set(negative)
        order = np.argsort(-sims)
        out = []
        for i in order:
            word = self._vocab[i]
            if word in excluded:
                continue
            out.append((word, float(sims[i])))
            if len(out) >= topn:
                break
        return out


def _load_embeddings():
    """Return (model, source_label). Never raises."""
    try:
        import gensim.downloader as api

        loaded = api.load("glove-wiki-gigaword-100")
        return loaded, "gensim glove-wiki-gigaword-100 (400k words, 100D)"
    except Exception as err:
        print(
            f"[fallback] Could not load GloVe ({type(err).__name__}: {err}). "
            f"Using a synthetic {30}-word dictionary with deterministic random vectors. "
            f"Nearest-neighbour rankings and analogies will not be meaningful."
        )
        return _SyntheticEmbedding(dim=32), "synthetic fallback (random vectors)"


model, _model_source = _load_embeddings()
print(f"Embedding source: {_model_source}")
print(f"Vocabulary size: {len(model.key_to_index):,}")


# --- Part 1: nearest neighbours ---------------------------------------------------
def _bar_color(score: float) -> str:
    """Map a similarity score in roughly [0.3, 0.8] to a blue-to-green gradient."""
    t = max(0.0, min(1.0, (score - 0.3) / 0.5))
    r = int(59 + t * (74 - 59))
    g = int(130 + t * (222 - 130))
    b = int(246 + t * (128 - 246))
    return f"rgb({r},{g},{b})"


def _render_neighbours(word: str, results) -> str:
    rows = []
    for w, s in results:
        pct = max(0, min(100, s * 100))
        col = _bar_color(s)
        rows.append(
            f'<tr>'
            f'<td style="padding:5px 12px;font-weight:600;color:#e2e8f0;font-size:14px;white-space:nowrap">{html_module.escape(w)}</td>'
            f'<td style="padding:5px 8px;width:100%">'
            f'<div style="background:{col};height:20px;border-radius:4px;width:{pct}%;'
            f'min-width:2px"></div></td>'
            f'<td style="padding:5px 12px;color:#94a3b8;font-size:13px;white-space:nowrap">{s:.3f}</td>'
            f'</tr>'
        )
    return (
        f'<div style="margin:8px 0 4px;font-size:15px;color:#e2e8f0">'
        f'10 nearest neighbours of <strong style="color:#60a5fa">{html_module.escape(word)}</strong></div>'
        f'<table style="width:100%;border-collapse:collapse">{"".join(rows)}</table>'
    )


_word_input = widgets.Text(
    value="doctor", placeholder="Type a word...",
    layout=widgets.Layout(width="300px"),
    style={"description_width": "0px"},
)
_btn_p1 = widgets.Button(
    description="Search", button_style="primary",
    layout=widgets.Layout(width="120px"),
)
_handle_p1 = None


def _render_search(word: str):
    if word not in model.key_to_index:
        return HTML(
            f'<div style="color:#f87171;padding:8px">'
            f'Word <strong>"{html_module.escape(word)}"</strong> not in vocabulary. '
            f'Try a common lowercase English word.</div>'
        )
    results = model.most_similar(word, topn=10)
    return HTML(_render_neighbours(word, results))


def _search(_=None):
    if _handle_p1 is None:
        return
    word = _word_input.value.strip().lower()
    if not word:
        return
    _handle_p1.update(_render_search(word))


_btn_p1.on_click(_search)
_word_input.on_submit(lambda _w: _search())

display(widgets.HTML('<h4 style="margin-top:4px">Part 1: nearest neighbours under cosine similarity</h4>'))
display(widgets.VBox([widgets.HBox([_word_input, _btn_p1])]))
_handle_p1 = display(_render_search(_word_input.value.strip().lower()), display_id="embeddings-part1")


In [ ]:
"""§3 Part 2: vector arithmetic for analogies.

Reuses the `model` object loaded in Part 1. Computes `pos1 + pos2 - neg` in
vector space and reports the closest five vocabulary entries.
"""

_pos1 = widgets.Text(value="king", placeholder="positive 1",
                     layout=widgets.Layout(width="160px"),
                     style={"description_width": "0px"})
_pos2 = widgets.Text(value="woman", placeholder="positive 2",
                     layout=widgets.Layout(width="160px"),
                     style={"description_width": "0px"})
_neg = widgets.Text(value="man", placeholder="negative",
                    layout=widgets.Layout(width="160px"),
                    style={"description_width": "0px"})
_btn_p2 = widgets.Button(
    description="Compute", button_style="primary",
    layout=widgets.Layout(width="130px"),
)
_handle_p2 = None


def _render_analogy(p1: str, p2: str, n: str):
    missing = [w for w in [p1, p2, n] if w not in model.key_to_index]
    if missing:
        return HTML(
            f'<div style="color:#f87171;padding:8px">'
            f'Not found: <strong>{", ".join(missing)}</strong>. '
            f'Try common lowercase English words.</div>'
        )
    results = model.most_similar(positive=[p1, p2], negative=[n], topn=5)
    if not results:
        return HTML('<div style="color:#94a3b8;padding:8px">No results.</div>')
    formula = (
        f'<div style="font-size:16px;color:#e2e8f0;margin:8px 0 12px">'
        f'<span style="color:#4ade80">{html_module.escape(p1)}</span> + '
        f'<span style="color:#4ade80">{html_module.escape(p2)}</span> &minus; '
        f'<span style="color:#f87171">{html_module.escape(n)}</span> &approx; '
        f'<strong style="color:#fbbf24;font-size:18px">{html_module.escape(results[0][0])}</strong>'
        f'<span style="color:#94a3b8;font-size:13px"> ({results[0][1]:.3f})</span>'
        f'</div>'
    )
    rows = "".join(
        f'<tr><td style="padding:4px 12px;color:{"#fbbf24" if i == 0 else "#e2e8f0"};'
        f'font-weight:{"700" if i == 0 else "400"};font-size:14px">{html_module.escape(w)}</td>'
        f'<td style="padding:4px 12px;color:#94a3b8;font-size:13px">{s:.3f}</td></tr>'
        for i, (w, s) in enumerate(results)
    )
    table = f'<table style="border-collapse:collapse">{rows}</table>'
    return HTML(formula + table)


def _analogy(_=None):
    if _handle_p2 is None:
        return
    p1 = _pos1.value.strip().lower()
    p2 = _pos2.value.strip().lower()
    n = _neg.value.strip().lower()
    if not (p1 and p2 and n):
        return
    _handle_p2.update(_render_analogy(p1, p2, n))


_btn_p2.on_click(_analogy)
for box in [_pos1, _pos2, _neg]:
    box.on_submit(lambda _w: _analogy())

display(widgets.HTML('<h4>Part 2: analogy arithmetic in vector space</h4>'))
display(widgets.VBox([
    widgets.HTML(
        '<div style="font-size:14px;color:#94a3b8;margin-bottom:4px">'
        'result &approx; <span style="color:#4ade80">positive1</span> + '
        '<span style="color:#4ade80">positive2</span> &minus; '
        '<span style="color:#f87171">negative</span></div>'
    ),
    widgets.HBox([
        widgets.VBox([widgets.HTML('<span style="color:#4ade80;font-size:12px">positive 1</span>'), _pos1]),
        widgets.VBox([widgets.HTML('<span style="color:#4ade80;font-size:12px">positive 2</span>'), _pos2]),
        widgets.VBox([widgets.HTML('<span style="color:#f87171;font-size:12px">negative</span>'), _neg]),
        _btn_p2,
    ]),
]))
_handle_p2 = display(
    _render_analogy(
        _pos1.value.strip().lower(),
        _pos2.value.strip().lower(),
        _neg.value.strip().lower(),
    ),
    display_id="embeddings-part2",
)


In [ ]:
"""§3 Part 3: bias probes along the gender axis.

For each profession the cell asks what the embedding returns when the
profession vector is shifted toward "woman" (positive=[prof, woman],
negative=[man]) and toward "man" (positive=[prof, man], negative=[woman]).
The asymmetries reflect biases present in the training corpus and
inherited by every downstream system built on these vectors.
"""

BIAS_PROBES = [
    "doctor", "nurse", "professor", "teacher", "engineer",
    "scientist", "programmer", "artist", "lawyer", "therapist",
    "surgeon", "receptionist", "pilot", "secretary", "athlete",
]


def _bias_probe(profession: str, gender_pos: str, gender_neg: str):
    """Return (word, score) for: profession + gender_pos - gender_neg.

    Skips outputs that are just the input word, its plural, or one of the
    axis words, since those are uninformative for the bias question.
    """
    try:
        results = model.most_similar(
            positive=[profession, gender_pos],
            negative=[gender_neg],
            topn=5,
        )
    except KeyError:
        return ("(missing)", 0.0)
    for w, s in results:
        if w not in (profession, profession + "s", gender_pos, gender_neg):
            return (w, s)
    return results[0] if results else ("(no result)", 0.0)


_rows_html = []
for prof in BIAS_PROBES:
    if prof not in model.key_to_index:
        continue
    fem_word, fem_score = _bias_probe(prof, "woman", "man")
    mas_word, mas_score = _bias_probe(prof, "man", "woman")
    fem_color = "#f472b6" if fem_word != prof else "#94a3b8"
    mas_color = "#60a5fa" if mas_word != prof else "#94a3b8"
    _rows_html.append(
        f'<tr style="border-bottom:1px solid #1e293b">'
        f'<td style="padding:6px 14px;font-weight:600;color:#e2e8f0;font-size:14px">{prof}</td>'
        f'<td style="padding:6px 14px;color:{fem_color};font-size:14px">{html_module.escape(fem_word)} '
        f'<span style="color:#475569;font-size:11px">({fem_score:.2f})</span></td>'
        f'<td style="padding:6px 14px;color:{mas_color};font-size:14px">{html_module.escape(mas_word)} '
        f'<span style="color:#475569;font-size:11px">({mas_score:.2f})</span></td>'
        f'</tr>'
    )

if _rows_html:
    _table_html = (
        '<table style="border-collapse:collapse;width:100%;font-size:14px;margin-top:8px">'
        '<thead><tr style="background:#0f172a">'
        '<th style="padding:8px 14px;text-align:left;color:#cbd5e1">Profession</th>'
        '<th style="padding:8px 14px;text-align:left;color:#f472b6">+ woman &minus; man</th>'
        '<th style="padding:8px 14px;text-align:left;color:#60a5fa">+ man &minus; woman</th>'
        '</tr></thead>'
        f'<tbody>{"".join(_rows_html)}</tbody></table>'
    )
    display(widgets.HTML('<h4>Part 3: gender associations encoded in the embedding</h4>'))
    display(HTML(
        '<div style="margin-bottom:8px;color:#94a3b8;font-size:13px">'
        'For each profession, the table shows the nearest neighbour after shifting along the gender axis. '
        'Words highlighted in colour are different from the input, which indicates the embedding associates '
        'them with the corresponding gender.</div>' + _table_html
    ))
else:
    display(widgets.HTML('<h4>Part 3: gender associations encoded in the embedding</h4>'))
    display(HTML(
        '<div style="color:#94a3b8;padding:8px">No probes available in the current vocabulary '
        '(fallback mode does not include all profession words).</div>'
    ))


# Open-ended probe along any axis.
_probe_word = widgets.Text(value="", placeholder="word to probe",
                           layout=widgets.Layout(width="260px"),
                           style={"description_width": "0px"})
_probe_axis1 = widgets.Text(value="man", placeholder="axis word 1",
                            layout=widgets.Layout(width="140px"),
                            style={"description_width": "0px"})
_probe_axis2 = widgets.Text(value="woman", placeholder="axis word 2",
                            layout=widgets.Layout(width="140px"),
                            style={"description_width": "0px"})
_btn_p3 = widgets.Button(
    description="Probe", button_style="warning",
    layout=widgets.Layout(width="110px"),
)
_handle_p3 = None


def _render_probe(word: str, a1: str, a2: str):
    if not (word and a1 and a2):
        return HTML(
            '<div style="color:#94a3b8;padding:8px;font-style:italic">'
            "Enter a word and two axis words above, then click Probe.</div>"
        )
    missing = [w for w in [word, a1, a2] if w not in model.key_to_index]
    if missing:
        return HTML(
            f'<div style="color:#f87171;padding:8px">Not found: '
            f'<strong>{", ".join(missing)}</strong></div>'
        )
    r1 = model.most_similar(positive=[word, a2], negative=[a1], topn=3)
    r2 = model.most_similar(positive=[word, a1], negative=[a2], topn=3)

    def _fmt(results):
        return ", ".join(f"{w} ({s:.3f})" for w, s in results) if results else "(empty)"

    return HTML(
        f'<div style="margin:8px 0;font-size:14px;color:#e2e8f0">'
        f'<strong>{html_module.escape(word)}</strong> + {html_module.escape(a2)} &minus; '
        f'{html_module.escape(a1)} &rarr; {_fmt(r1)}</div>'
        f'<div style="font-size:14px;color:#e2e8f0">'
        f'<strong>{html_module.escape(word)}</strong> + {html_module.escape(a1)} &minus; '
        f'{html_module.escape(a2)} &rarr; {_fmt(r2)}</div>'
    )


def _run_probe(_=None):
    if _handle_p3 is None:
        return
    word = _probe_word.value.strip().lower()
    a1 = _probe_axis1.value.strip().lower()
    a2 = _probe_axis2.value.strip().lower()
    if not (word and a1 and a2):
        return
    _handle_p3.update(_render_probe(word, a1, a2))


_btn_p3.on_click(_run_probe)
_probe_word.on_submit(lambda _w: _run_probe())

display(widgets.HTML('<h4>Open-ended probe along an arbitrary axis</h4>'))
display(widgets.VBox([
    widgets.HTML(
        '<div style="color:#94a3b8;font-size:13px;margin-bottom:4px">'
        'Pick any axis. Examples: rich vs poor, young vs old, urban vs rural.</div>'
    ),
    widgets.HBox([
        widgets.VBox([widgets.HTML('<span style="color:#e2e8f0;font-size:12px">word</span>'), _probe_word]),
        widgets.VBox([widgets.HTML('<span style="color:#94a3b8;font-size:12px">axis word 1</span>'), _probe_axis1]),
        widgets.VBox([widgets.HTML('<span style="color:#94a3b8;font-size:12px">axis word 2</span>'), _probe_axis2]),
        _btn_p3,
    ]),
]))
_handle_p3 = display(_render_probe("", "", ""), display_id="embeddings-probe")


## §4 Warm-ups

Two short exercises. The first implements cosine similarity, the primitive that powers every "nearest neighbour" lookup in §3. The second builds the input layer of the transformer: a `TokenEmbedder` that turns a batch of token IDs into a tensor with both content and position encoded.


In [ ]:
"""§4 Warm-up 1 (exercise): cosine similarity for two tensors.

Implement `cosine(a, b)` so it returns the cosine similarity between the two
inputs along their last dimension. The function must support broadcasting,
e.g. comparing one query vector against a batch of candidates.
"""

import torch


def cosine(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:
    """Cosine similarity along the last dim.

    Hint: dot product divided by the product of norms.
    torch.nn.functional has it as one line.
    """
    # TODO: implement
    raise NotImplementedError


u = torch.tensor([1.0, 0.0, 0.0])
v = torch.tensor([0.0, 1.0, 0.0])
w = torch.tensor([1.0, 0.0, 0.0])
# Expected: 0.000, 1.000
# After implementing, uncomment:
# print(f"cos(u, v) = {cosine(u, v).item():.3f}")
# print(f"cos(u, w) = {cosine(u, w).item():.3f}")

try:
    cosine(u, v)
except NotImplementedError:
    print("cosine not implemented yet.")


In [ ]:
"""§4 Warm-up 2 (exercise): TokenEmbedder, the input layer of a transformer.

Build `TokenEmbedder` so that an input of shape `(batch, seq)` returns a tensor
of shape `(batch, seq, d_model)`: token embedding plus sinusoidal positional
encoding. This module is the input layer of `TinyGPT` in notebook 1c.
"""

import math

import torch
from torch import nn


def sinusoidal_pe(max_len: int, d_model: int) -> torch.Tensor:
    """Helper provided. Returns a (max_len, d_model) tensor of sin/cos position codes."""
    position = torch.arange(max_len).unsqueeze(1).float()
    div_term = torch.exp(
        torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
    )
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class TokenEmbedder(nn.Module):
    """Token embedding plus sinusoidal positional encoding.

    For an input of shape (batch, seq) return shape (batch, seq, d_model).
    """

    def __init__(self, vocab_size: int, d_model: int, max_seq_len: int = 1024):
        super().__init__()
        # TODO: instantiate nn.Embedding(vocab_size, d_model)
        # TODO: register the positional encoding as a buffer named "pos"
        raise NotImplementedError

    def forward(self, ids: torch.Tensor) -> torch.Tensor:
        # TODO: token embedding + positional encoding sliced to current seq length
        raise NotImplementedError


# Expected: torch.Size([2, 8, 16])
# torch.manual_seed(0)
# emb = TokenEmbedder(vocab_size=50, d_model=16, max_seq_len=32)
# ids = torch.randint(0, 50, (2, 8))
# print(emb(ids).shape)

try:
    TokenEmbedder(vocab_size=50, d_model=16, max_seq_len=32)
except NotImplementedError:
    print("TokenEmbedder not implemented yet.")


The `TokenEmbedder` above is the input layer of `TinyGPT` in notebook 1c. The body of the transformer (attention blocks, residual stream, output projection) sits on top of exactly this tensor shape `(batch, seq, d_model)`. Position information is baked in once, here, and then never re-injected.


## §6 Recap and next step

You have inspected a trained embedding space (cosine neighbours, analogy arithmetic, gender bias on profession words), implemented cosine similarity in PyTorch, and built `TokenEmbedder`: a module that sums a token vector with a sinusoidal positional vector to produce the `(batch, seq, d_model)` tensor every transformer block expects.

The next notebook (1c) reuses `TokenEmbedder` as the input layer of `TinyGPT` and builds the attention, multi-head, and transformer-block sublayers that sit on top of it.
